# Customer Segmentation — Clustering Notebook

## Table of Contents

1. [Setup & Data Loading](#1-setup--data-loading)
2. [Clustering with Standard Scaler](#2-clustering-with-standard-scaler)
   - 2.1 K-Means
   - 2.2 Mean Shift
   - 2.3 Hierarchical Clustering
   - 2.4 DBSCAN
   - 2.5 SOM (Self-Organizing Map)
3. [Clustering with Robust Scaler](#3-clustering-with-robust-scaler)
   - 3.1 K-Means
   - 3.2 Mean Shift
   - 3.3 Hierarchical Clustering
   - 3.4 DBSCAN
   - 3.5 SOM (Self-Organizing Map)
4. [Clustering with MinMax Scaler](#4-clustering-with-minmax-scaler)
   - 4.1 K-Means
   - 4.2 Mean Shift
   - 4.3 Hierarchical Clustering
   - 4.4 DBSCAN
   - 4.5 SOM (Self-Organizing Map)
5. [Model Comparison](#5-model-comparison)

# 1. Setup & Data Loading

In [ ]:
# Import necessary libraries for data processing and visualization
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans, MeanShift
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import importlib

# Import all clustering algorithms and helper functions from Clustering.py
import Clustering
importlib.reload(Clustering)
from Clustering import (
    find_optimal_k, fit_kmeans, fit_hierarchical, find_optimal_eps, 
    fit_dbscan, compare_models, fit_som, assign_som_clusters, 
    fit_meanshift, plot_umap, plot_tsne
)

# Import preprocessing functions from Preprocessing.py
from Preprocessing import (
    preprocess_data_standardscaler, preprocess_data_robustscaler, 
    preprocess_data_minmaxscaler, cluster_analysis
)

In [ ]:
# Load the engineered customer dataset
df = pd.read_csv('customer_info_engineered.csv')

# Compute total kids (Option C: Perfil Familiar e Demográfico)
df['total_kids'] = df['kids_home'] + df['teens_home']

# Define the selected subset of features for clustering (Family & Demographics)
option_c_features = [
    'total_kids', 
    'age', 
    'lifetime_total_distinct_products',
    'year_first_transaction'
]

# Filter the dataset before calling the preprocessing functions to prevent KNNImputer from hanging
df_filtered = df[option_c_features].copy()

# Preprocess and scale the dataset using Standard, Robust, and MinMax scalers
df_processed_st = preprocess_data_standardscaler(df_filtered.copy())
df_processed_rb = preprocess_data_robustscaler(df_filtered.copy())
df_processed_mm = preprocess_data_minmaxscaler(df_filtered.copy())

# Setup standard features datasets
df_saude_st = df_processed_st
df_saude_rb = df_processed_rb
df_saude_mm = df_processed_mm

# Run baseline profile analysis on the unscaled customer dataset (using fast median imputation)
df_profile = df.copy()
total_spend_prof = df_profile['total_lifetime_spend'].replace(0, np.nan)
df_profile['pct_vegetables'] = (df_profile['lifetime_spend_vegetables'] / total_spend_prof).fillna(0)
df_profile['pct_meat'] = (df_profile['lifetime_spend_meat'] / total_spend_prof).fillna(0)
df_profile['pct_fish'] = (df_profile['lifetime_spend_fish'] / total_spend_prof).fillna(0)
df_profile['pct_alcohol'] = (df_profile['lifetime_spend_alcohol_drinks'] / total_spend_prof).fillna(0)
df_profile['pct_videogames'] = (df_profile['lifetime_spend_videogames'] / total_spend_prof).fillna(0)

df_cluster_analysis = cluster_analysis(df_profile)

# 2. Clustering with Standard Scaler

## 2.1 K-Means

In [ ]:
# Search for the optimal number of clusters (k) using Elbow & Silhouette methods
inertias, silhouette_scores = find_optimal_k(df_saude_st)

In [ ]:
# Fit the final K-Means model with the optimal k=4 clusters
kmeans_profile_st = fit_kmeans(df_saude_st, 4)
print(kmeans_profile_st)

In [ ]:
# Fit KMeans with optimal k=4 to get cluster labels for UMAP & t-SNE visualization
kmeans_opt_st = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans_st = kmeans_opt_st.fit_predict(df_saude_st)

# Visualize the KMeans cluster assignments using UMAP & t-SNE dimensionality reduction
plot_umap(df_saude_st, labels_kmeans_st, title="UMAP - KMeans (Standard Scaler, k=4)")
plot_tsne(df_saude_st, labels_kmeans_st, title="t-SNE - KMeans (Standard Scaler, k=4)")

## 2.2 Mean Shift

In [ ]:
# Fit Mean Shift clustering to automatically determine the number of clusters
mean_shift_st = MeanShift(bin_seeding=True)
labels_ms_st = mean_shift_st.fit_predict(df_saude_st)
print("Standard Scaler MeanShift cluster count:", len(np.unique(labels_ms_st)))
if len(np.unique(labels_ms_st)) > 1:
    print(f"Standard Scaler MeanShift silhouette: {silhouette_score(df_saude_st, labels_ms_st):.4f}")
else:
    print("Standard Scaler MeanShift silhouette: not defined (single cluster)")
print(pd.Series(labels_ms_st).value_counts().sort_index())

In [ ]:
# Visualize Mean Shift cluster assignments using UMAP & t-SNE
plot_umap(df_saude_st, labels_ms_st, title="UMAP - Mean Shift (Standard Scaler)")
plot_tsne(df_saude_st, labels_ms_st, title="t-SNE - Mean Shift (Standard Scaler)")

## 2.3 Hierarchical Clustering

In [ ]:
# Fit the Hierarchical Agglomerative Clustering model with 5 clusters (Ward's linkage)
hierarchical_model_st, labels_hierarchical_st = fit_hierarchical(df_saude_st, 5, method='ward')

In [ ]:
# Visualize Hierarchical Clustering cluster assignments using UMAP & t-SNE
plot_umap(df_saude_st, labels_hierarchical_st, title="UMAP - Hierarchical Clustering (Standard Scaler, k=5)")
plot_tsne(df_saude_st, labels_hierarchical_st, title="t-SNE - Hierarchical Clustering (Standard Scaler, k=5)")

## 2.4 DBSCAN

In [ ]:
# Generate a K-Distance graph to determine the optimal epsilon (eps) value for DBSCAN
find_optimal_eps(df_saude_st, n_neighbors=5)

In [ ]:
# Fit DBSCAN clustering using epsilon=3.0 and min_samples=5
dbscan_model_st, labels_dbscan_st = fit_dbscan(df_saude_st, 3.0, min_samples=5)

# Visualize DBSCAN (eps=3.0) cluster assignments using UMAP & t-SNE
plot_umap(df_saude_st, labels_dbscan_st, title="UMAP - DBSCAN (Standard Scaler, eps=3.0)")
plot_tsne(df_saude_st, labels_dbscan_st, title="t-SNE - DBSCAN (Standard Scaler, eps=3.0)")

## 2.5 Self-Organizing Map (SOM)

In [ ]:
# Fit the Self-Organizing Map (SOM) to the dataset
som_result_st = fit_som(df_saude_st, map_shape=(10, 10), n_iterations=1000, learning_rate=0.5, random_state=42, plot_u_matrix=False)

In [ ]:
# Assign each data sample to a SOM neuron cluster label
labels_som_st, neuron_labels_st = assign_som_clusters(som_result_st['weights'], df_saude_st, n_clusters=5, random_state=42)

# Visualize SOM cluster assignments using UMAP & t-SNE
plot_umap(df_saude_st, labels_som_st, title="UMAP - SOM (Standard Scaler, k=5)")
plot_tsne(df_saude_st, labels_som_st, title="t-SNE - SOM (Standard Scaler, k=5)")

# 3. Clustering with Robust Scaler

## 3.1 K-Means

In [ ]:
# Search for the optimal number of clusters (k) using Elbow & Silhouette methods
inertias, silhouette_scores = find_optimal_k(df_saude_rb)

In [ ]:
# Fit the final K-Means model with the optimal k=4 clusters
kmeans_profile_rb = fit_kmeans(df_saude_rb, 4)
print(kmeans_profile_rb)

In [ ]:
# Fit KMeans with optimal k=4 to get cluster labels for UMAP & t-SNE visualization
kmeans_opt_rb = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans_rb = kmeans_opt_rb.fit_predict(df_saude_rb)

# Visualize the KMeans cluster assignments using UMAP & t-SNE
plot_umap(df_saude_rb, labels_kmeans_rb, title="UMAP - KMeans (Robust Scaler, k=4)")
plot_tsne(df_saude_rb, labels_kmeans_rb, title="t-SNE - KMeans (Robust Scaler, k=4)")

## 3.2 Mean Shift

In [ ]:
# Fit Mean Shift clustering to automatically determine the number of clusters
mean_shift_rb = MeanShift(bin_seeding=True)
labels_ms_rb = mean_shift_rb.fit_predict(df_saude_rb)
print("Robust Scaler MeanShift cluster count:", len(np.unique(labels_ms_rb)))
if len(np.unique(labels_ms_rb)) > 1:
    print(f"Robust Scaler MeanShift silhouette: {silhouette_score(df_saude_rb, labels_ms_rb):.4f}")
else:
    print("Robust Scaler MeanShift silhouette: not defined (single cluster)")
print(pd.Series(labels_ms_rb).value_counts().sort_index())

In [ ]:
# Visualize Mean Shift cluster assignments using UMAP & t-SNE
plot_umap(df_saude_rb, labels_ms_rb, title="UMAP - Mean Shift (Robust Scaler)")
plot_tsne(df_saude_rb, labels_ms_rb, title="t-SNE - Mean Shift (Robust Scaler)")

## 3.3 Hierarchical Clustering

In [ ]:
# Fit the Hierarchical Agglomerative Clustering model with 5 clusters (Ward's linkage)
hierarchical_model_rb, labels_hierarchical_rb = fit_hierarchical(df_saude_rb, 5, method='ward')

In [ ]:
# Visualize Hierarchical Clustering cluster assignments using UMAP & t-SNE
plot_umap(df_saude_rb, labels_hierarchical_rb, title="UMAP - Hierarchical Clustering (Robust Scaler, k=5)")
plot_tsne(df_saude_rb, labels_hierarchical_rb, title="t-SNE - Hierarchical Clustering (Robust Scaler, k=5)")

## 3.4 DBSCAN

In [ ]:
# Generate a K-Distance graph to determine the optimal epsilon (eps) value for DBSCAN
find_optimal_eps(df_saude_rb, n_neighbors=5)

In [ ]:
# Fit DBSCAN clustering using epsilon=3.0 and min_samples=5
dbscan_model_rb, labels_dbscan_rb = fit_dbscan(df_saude_rb, 3.0, min_samples=5)

# Visualize DBSCAN (eps=3.0) cluster assignments using UMAP & t-SNE
plot_umap(df_saude_rb, labels_dbscan_rb, title="UMAP - DBSCAN (Robust Scaler, eps=3.0)")
plot_tsne(df_saude_rb, labels_dbscan_rb, title="t-SNE - DBSCAN (Robust Scaler, eps=3.0)")

## 3.5 Self-Organizing Map (SOM)

In [ ]:
# Fit the Self-Organizing Map (SOM) to the dataset
som_result_rb = fit_som(df_saude_rb, map_shape=(10, 10), n_iterations=1000, learning_rate=0.5, random_state=42, plot_u_matrix=False)

In [ ]:
# Assign each data sample to a SOM neuron cluster label
labels_som_rb, neuron_labels_rb = assign_som_clusters(som_result_rb['weights'], df_saude_rb, n_clusters=5, random_state=42)

# Visualize SOM cluster assignments using UMAP & t-SNE
plot_umap(df_saude_rb, labels_som_rb, title="UMAP - SOM (Robust Scaler, k=5)")
plot_tsne(df_saude_rb, labels_som_rb, title="t-SNE - SOM (Robust Scaler, k=5)")

# 4. Clustering with MinMax Scaler

## 4.1 K-Means

In [ ]:
# Search for the optimal number of clusters (k) using Elbow & Silhouette methods
inertias, silhouette_scores = find_optimal_k(df_saude_mm)

In [ ]:
# Fit the final K-Means model with the optimal k=4 clusters
kmeans_profile_mm = fit_kmeans(df_saude_mm, 4)
print(kmeans_profile_mm)

In [ ]:
# Fit KMeans with optimal k=4 to get cluster labels for UMAP & t-SNE visualization
kmeans_opt_mm = KMeans(n_clusters=4, random_state=42, n_init=10)
labels_kmeans_mm = kmeans_opt_mm.fit_predict(df_saude_mm)

# Visualize the KMeans cluster assignments using UMAP & t-SNE
plot_umap(df_saude_mm, labels_kmeans_mm, title="UMAP - KMeans (MinMax Scaler, k=4)")
plot_tsne(df_saude_mm, labels_kmeans_mm, title="t-SNE - KMeans (MinMax Scaler, k=4)")

## 4.2 Mean Shift

In [ ]:
# Fit Mean Shift clustering to automatically determine the number of clusters
mean_shift_mm = MeanShift(bin_seeding=True)
labels_ms_mm = mean_shift_mm.fit_predict(df_saude_mm)
print("MinMax Scaler MeanShift cluster count:", len(np.unique(labels_ms_mm)))
if len(np.unique(labels_ms_mm)) > 1:
    print(f"MinMax Scaler MeanShift silhouette: {silhouette_score(df_saude_mm, labels_ms_mm):.4f}")
else:
    print("MinMax Scaler MeanShift silhouette: not defined (single cluster)")
print(pd.Series(labels_ms_mm).value_counts().sort_index())

In [ ]:
# Visualize Mean Shift cluster assignments using UMAP & t-SNE
plot_umap(df_saude_mm, labels_ms_mm, title="UMAP - Mean Shift (MinMax Scaler)")
plot_tsne(df_saude_mm, labels_ms_mm, title="t-SNE - Mean Shift (MinMax Scaler)")

## 4.3 Hierarchical Clustering

In [ ]:
# Fit the Hierarchical Agglomerative Clustering model with 5 clusters (Ward's linkage)
hierarchical_model_mm, labels_hierarchical_mm = fit_hierarchical(df_saude_mm, 5, method='ward')

In [ ]:
# Visualize Hierarchical Clustering cluster assignments using UMAP & t-SNE
plot_umap(df_saude_mm, labels_hierarchical_mm, title="UMAP - Hierarchical Clustering (MinMax Scaler, k=5)")
plot_tsne(df_saude_mm, labels_hierarchical_mm, title="t-SNE - Hierarchical Clustering (MinMax Scaler, k=5)")

## 4.4 DBSCAN

In [ ]:
# Generate a K-Distance graph to determine the optimal epsilon (eps) value for DBSCAN
find_optimal_eps(df_saude_mm, n_neighbors=5)

In [ ]:
# Fit DBSCAN clustering using epsilon=3.0 and min_samples=5
dbscan_model_mm, labels_dbscan_mm = fit_dbscan(df_saude_mm, 3.0, min_samples=5)

# Visualize DBSCAN (eps=3.0) cluster assignments using UMAP & t-SNE
plot_umap(df_saude_mm, labels_dbscan_mm, title="UMAP - DBSCAN (MinMax Scaler, eps=3.0)")
plot_tsne(df_saude_mm, labels_dbscan_mm, title="t-SNE - DBSCAN (MinMax Scaler, eps=3.0)")

## 4.5 Self-Organizing Map (SOM)

In [ ]:
# Fit the Self-Organizing Map (SOM) to the dataset
som_result_mm = fit_som(df_saude_mm, map_shape=(10, 10), n_iterations=1000, learning_rate=0.5, random_state=42, plot_u_matrix=False)

In [ ]:
# Assign each data sample to a SOM neuron cluster label
labels_som_mm, neuron_labels_mm = assign_som_clusters(som_result_mm['weights'], df_saude_mm, n_clusters=5, random_state=42)

# Visualize SOM cluster assignments using UMAP & t-SNE
plot_umap(df_saude_mm, labels_som_mm, title="UMAP - SOM (MinMax Scaler, k=5)")
plot_tsne(df_saude_mm, labels_som_mm, title="t-SNE - SOM (MinMax Scaler, k=5)")

# 5. Model Comparison

In [ ]:
# Compare all fitted models using their Silhouette Scores to find the best configuration
labels_dict_st = {
    'KMeans (k=4)': labels_kmeans_st,
    'Hierarchical (k=5)': labels_hierarchical_st,
    'DBSCAN (eps=3.0)': labels_dbscan_st,
    'MeanShift': labels_ms_st,
    'SOM (k=5)': labels_som_st
}
print("--- Standard Scaler Comparison ---")
compare_models(df_saude_st, labels_dict_st)

# Compare all fitted models for Robust Scaler
labels_dict_rb = {
    'KMeans (k=4)': labels_kmeans_rb,
    'Hierarchical (k=5)': labels_hierarchical_rb,
    'DBSCAN (eps=3.0)': labels_dbscan_rb,
    'MeanShift': labels_ms_rb,
    'SOM (k=5)': labels_som_rb
}
print("\n--- Robust Scaler Comparison ---")
compare_models(df_saude_rb, labels_dict_rb)

# Compare all fitted models for MinMax Scaler
labels_dict_mm = {
    'KMeans (k=4)': labels_kmeans_mm,
    'Hierarchical (k=5)': labels_hierarchical_mm,
    'DBSCAN (eps=3.0)': labels_dbscan_mm,
    'MeanShift': labels_ms_mm,
    'SOM (k=5)': labels_som_mm
}
print("\n--- MinMax Scaler Comparison ---")
compare_models(df_saude_mm, labels_dict_mm)